# Lab 03 – Load rail reference data into the Lakehouse

Downloads **CORPUS** and **SMART** from Network Rail Open Data (authenticated) and **NaPTAN** rail stations
(Open Government Licence v3.0), then writes two Delta tables to the attached Lakehouse:

| Table | Purpose |
|---|---|
| `locations` | STANOX / TIPLOC / CRS / NLC → location name, latitude, longitude |
| `smart_berths` | TD berth → STANOX stepping data (SMART) |

**Before you run:** attach `RailLakehouse` as the default lakehouse, and either set `KEY_VAULT_URL`
(recommended – secrets `nrod-username`/`nrod-password`) or type credentials into the parameters cell
*for this session only* – never save them in the notebook.

Data: contains Network Rail Infrastructure Limited data licensed under the Open Government Licence; NaPTAN © Crown copyright (OGL v3.0).

In [ ]:
# Parameters
KEY_VAULT_URL = ""            # e.g. https://<vault>.vault.azure.net/  (from Lab 01 step 8) - leave empty to use the values below
NROD_USERNAME_SECRET = "nrod-username"
NROD_PASSWORD_SECRET = "nrod-password"
NROD_USERNAME = ""            # only if not using Key Vault - do NOT save the notebook with values here
NROD_PASSWORD = ""
INCLUDE_NAPTAN = True

In [ ]:
if KEY_VAULT_URL:
    import notebookutils
    NROD_USERNAME = notebookutils.credentials.getSecret(KEY_VAULT_URL, NROD_USERNAME_SECRET)
    NROD_PASSWORD = notebookutils.credentials.getSecret(KEY_VAULT_URL, NROD_PASSWORD_SECRET)
assert NROD_USERNAME and NROD_PASSWORD, "Set KEY_VAULT_URL or NROD_USERNAME/NROD_PASSWORD"

In [ ]:
import csv, gzip, io, json
import requests
import pandas as pd

NROD_BASE = "https://publicdatafeeds.networkrail.co.uk/ntrod/SupportingFileAuthenticate?type="
NAPTAN_URL = "https://naptan.api.dft.gov.uk/v1/access-nodes?dataFormat=csv"

def maybe_gunzip(raw: bytes) -> bytes:
    return gzip.decompress(raw) if raw[:2] == b"\x1f\x8b" else raw

def first_list(doc):
    if isinstance(doc, list):
        return doc
    for v in doc.values():
        if isinstance(v, list):
            return v
    raise ValueError("No array found")

def clean(v):
    return "" if v is None else str(v).strip()

def download_nrod(dataset):
    r = requests.get(NROD_BASE + dataset, auth=(NROD_USERNAME, NROD_PASSWORD), timeout=300)
    r.raise_for_status()
    return first_list(json.loads(maybe_gunzip(r.content)))

corpus = download_nrod("CORPUS")
smart = download_nrod("SMART")
print(len(corpus), "CORPUS rows;", len(smart), "SMART rows")

In [ ]:
coords = {}
if INCLUDE_NAPTAN:
    r = requests.get(NAPTAN_URL, timeout=600)
    r.raise_for_status()
    for row in csv.DictReader(io.StringIO(r.content.decode("utf-8-sig"))):
        atco = row.get("ATCOCode", "")
        if row.get("StopType") == "RLY" and atco.startswith("9100"):
            try:
                coords[atco[4:]] = (float(row["Latitude"]), float(row["Longitude"]))
            except (KeyError, ValueError):
                pass
print(len(coords), "rail stations with coordinates")

In [ ]:
rows = []
for c in corpus:
    stanox, tiploc = clean(c.get("STANOX")), clean(c.get("TIPLOC"))
    if not stanox and not tiploc:
        continue
    lat, lon = coords.get(tiploc, (None, None))
    rows.append({"stanox": stanox, "tiploc": tiploc, "crs": clean(c.get("3ALPHA")), "nlc": clean(c.get("NLC")),
                 "name": clean(c.get("NLCDESC")), "lat": lat, "lon": lon})

locations = pd.DataFrame(rows, columns=["stanox", "tiploc", "crs", "nlc", "name", "lat", "lon"])
locations["lat"] = locations["lat"].astype("float64")
locations["lon"] = locations["lon"].astype("float64")
spark.createDataFrame(locations).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("locations")

smart_df = pd.DataFrame([{k: clean(v) for k, v in r.items()} for r in smart])
spark.createDataFrame(smart_df).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("smart_berths")
print("Tables written: locations, smart_berths")

## Checkpoint

In [ ]:
display(spark.sql("SELECT COUNT(*) AS rows, COUNT(lat) AS with_coords FROM locations"))
display(spark.sql("SELECT * FROM locations WHERE crs IN ('KGX','LDS','EDB','MAN') ORDER BY crs"))